# Day 52：指令微调数据集构建（Alpaca / ShareGPT、清洗去重与 Self-Instruct）

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第五阶段（Stage 5）**Day 52**！今天我们将实战微调数据工程全链路：基于 Pydantic v2 定义 **Alpaca / ShareGPT** 双格式 Schema 并实现双向无损转换，验证**空白规范化、最小长度过滤、SHA-256 精确去重与字符 N-gram Jaccard 近似去重**，最后合成 **500 条企业级智能客服指令数据集**并完成 JSONL 导出与回读校验（`v0.5.0-day52`）。

## 1. 环境初始化与导入 `project_iterations/day_52` 模块

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day52_dir = repo_root / "project_iterations" / "day_52"
if str(day52_dir) not in sys.path:
    sys.path.insert(0, str(day52_dir))

from zhishu_agent import (
    AlpacaSample,
    ShareGPTSample,
    alpaca_to_sharegpt,
    clean_and_deduplicate,
    clean_and_deduplicate_with_stats,
    compute_dataset_stats,
    export_jsonl,
    generate_customer_service_dataset,
    jaccard_similarity,
    load_jsonl,
    sharegpt_to_alpaca,
)

print("已加载模块路径:", day52_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_52


## 2. Alpaca & ShareGPT 双格式 Schema 与双向无损转换

In [2]:
sample = AlpacaSample(
    instruction="查询跨月增值税专用发票红冲流程",
    input="发票单号：INV20263008 | 金额：3650元",
    output="您好！已跨月的数电专票需在「发票管理 -> 申请红冲/换开」提交红字冲销申请，1 个工作日内为您重开蓝字发票。",
    system="你是智枢企业级智能客服助手。",
    category="invoice_billing",
)
sg_sample = alpaca_to_sharegpt(sample)
print("ShareGPT JSON:\n", sg_sample.model_dump_json(by_alias=True, indent=2))

restored = sharegpt_to_alpaca(sg_sample)
assert restored == sample
print("双向无损转换校验通过！")

ShareGPT JSON:
 {
  "conversations": [
    {
      "from": "human",
      "value": "查询跨月增值税专用发票红冲流程\n\n输入：发票单号：INV20263008 | 金额：3650元"
    },
    {
      "from": "gpt",
      "value": "您好！已跨月的数电专票需在「发票管理 -> 申请红冲/换开」提交红字冲销申请，1 个工作日内为您重开蓝字发票。"
    }
  ],
  "system": "你是智枢企业级智能客服助手。",
  "category": "invoice_billing"
}
双向无损转换校验通过！


## 3. 文本清洗、最小长度过滤与两级去重（Hash + Jaccard）

In [3]:
dirty_batch = generate_customer_service_dataset(count=100, seed=7, include_dirty_ratio=0.2)
cleaned_batch, dedup_stats = clean_and_deduplicate_with_stats(dirty_batch)
print("含 20% 脏数据批次清洗统计:", dedup_stats.model_dump())
assert dedup_stats.output_count == 80

含 20% 脏数据批次清洗统计: {'input_count': 100, 'filtered_short_count': 0, 'exact_duplicate_count': 10, 'near_duplicate_count': 10, 'output_count': 80}


## 4. Self-Instruct 合成 500 条客服数据集、统计与 JSONL 导出回读

In [4]:
import tempfile

raw_500 = generate_customer_service_dataset(count=500, seed=42)
cleaned_500, stats_500 = clean_and_deduplicate_with_stats(raw_500)
ds_stats = compute_dataset_stats(cleaned_500)

print("500 条合成数据集去重统计:", stats_500.model_dump())
print("500 条数据集分布与长度统计:", ds_stats.model_dump())
assert ds_stats.total_count == 500

sharegpt_500 = [alpaca_to_sharegpt(s) for s in cleaned_500]
with tempfile.TemporaryDirectory() as tmp:
    tmp_dir = Path(tmp)
    alpaca_path = export_jsonl(cleaned_500, tmp_dir / "cs_alpaca_500.jsonl")
    sharegpt_path = export_jsonl(sharegpt_500, tmp_dir / "cs_sharegpt_500.jsonl")
    reloaded_a = load_jsonl(alpaca_path, schema="alpaca")
    reloaded_s = load_jsonl(sharegpt_path, schema="sharegpt")
    assert len(reloaded_a) == 500 and len(reloaded_s) == 500
    print(f"JSONL 导出与回读验证成功: Alpaca={len(reloaded_a)} 条, ShareGPT={len(reloaded_s)} 条")

500 条合成数据集去重统计: {'input_count': 500, 'filtered_short_count': 0, 'exact_duplicate_count': 0, 'near_duplicate_count': 0, 'output_count': 500}
500 条数据集分布与长度统计: {'total_count': 500, 'category_distribution': {'account_security': 100, 'invoice_billing': 100, 'order_logistics': 100, 'refund_aftersales': 100, 'vip_membership': 100}, 'avg_instruction_len': 100.58, 'avg_input_len': 20.85, 'avg_output_len': 109.48, 'with_input_ratio': 0.56}
JSONL 导出与回读验证成功: Alpaca=500 条, ShareGPT=500 条
